# 05 — LoRA Eval
ROUGE-L / BERTScore box plots by level; base vs fine-tuned comparison.

In [ ]:
import json, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from scholar.generation.eval import _load_scitldr, _rouge_l

records = _load_scitldr(limit=100)
print(f'Loaded {len(records)} SciTLDR samples')
records[:2]

In [ ]:
from scholar.generation.inference import Phi3Generator

levels = ['undergrad', 'grad', 'researcher']
results = {level: {'base': [], 'lora': []} for level in levels}
gen = Phi3Generator.get_instance()

for level in levels:
    for rec in records[:30]:  # subset for speed
        try:
            out = gen.generate(rec['title'], rec['abstract'], 'Evaluation.', level)
            score = _rouge_l(out.get('summary',''), rec['reference'])
            results[level]['base'].append(score)
        except Exception:
            results[level]['base'].append(0.0)

print('Base model evaluation done.')
for level in levels:
    print(f'  {level}: ROUGE-L={np.mean(results[level]["base"]):.4f}')

In [ ]:
lora_path = Path('./data/lora_adapter')
if lora_path.exists():
    Phi3Generator._instance = None  # reload with adapter
    gen = Phi3Generator.get_instance()
    for level in levels:
        for rec in records[:30]:
            try:
                out = gen.generate(rec['title'], rec['abstract'], 'Evaluation.', level)
                score = _rouge_l(out.get('summary',''), rec['reference'])
                results[level]['lora'].append(score)
            except Exception:
                results[level]['lora'].append(0.0)
    print('LoRA model evaluation done.')
else:
    print('LoRA adapter not found — run train_lora.py first')
    for level in levels:
        results[level]['lora'] = results[level]['base']

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5), sharey=True)
for ax, level in zip(axes, levels):
    data = [results[level]['base'], results[level]['lora']]
    bp = ax.boxplot(data, labels=['Base Phi-3', 'Phi-3 + LoRA'], patch_artist=True)
    for patch, color in zip(bp['boxes'], ['#ff9999','#99ccff']):
        patch.set_facecolor(color)
    ax.set_title(f'Level: {level}')
    ax.set_ylabel('ROUGE-L' if level == 'undergrad' else '')
    ax.grid(True, alpha=0.3)
plt.suptitle('ROUGE-L by Level: Base vs LoRA Fine-tuned')
plt.tight_layout()
plt.savefig('docs/lora_eval.png', dpi=150, bbox_inches='tight')
plt.show()